# 02 — Risk + Global Orchestrator pure-logic

Both `RiskAgent.calculate_quality_score / get_priority_queue` and `GlobalOrchestrator._filter_ready_items / _check_epic_complete / _uat_passes_g10` are deterministic Python — no LLM, no MCP. Feed synthetic Linear state, watch the outputs.

Why this exists: the pieces of the system you most need to **trust** under load are the deterministic ones. Easier to reason about when you can twist inputs and watch the queue change.

**Runtime-agnostic.** None of the cells below reach a runtime — they exercise pure functions on the agent classes. The same outputs hold whether the rest of the system is wired to Claude or Codex.


In [ ]:
from _helpers import ensure_src_on_path

ensure_src_on_path()
from hsb.agents.global_orchestrator import _uat_passes_g10
from hsb.agents.risk_agent import RiskAgent
from hsb.contracts.uat import UATResult, UATScenario

ra = RiskAgent()

## Quality score formula (skill 12)

`score = max(0, 100 - 10·qa_failures - 5·fix_subtasks - (15 if uat_failed) - 5·rework)`

Concrete examples — the breakdown dict shows you which penalty contributed what.


In [ ]:
examples = [
    ("clean", {"id": "LIN-1", "fix_subtask_count": 0, "qa_cycle_count": 0}, [], []),
    (
        "one-qa-fail",
        {"id": "LIN-2", "fix_subtask_count": 1, "qa_cycle_count": 1},
        [{"status": "changes_required"}],
        [],
    ),
    (
        "uat-failed",
        {"id": "LIN-3", "fix_subtask_count": 2, "qa_cycle_count": 2},
        [{"status": "changes_required"}],
        [{"overall_status": "changes_required"}],
    ),
    (
        "floor",
        {"id": "LIN-4", "fix_subtask_count": 30, "qa_cycle_count": 10},
        [{"status": "changes_required"}] * 10,
        [{"overall_status": "changes_required"}],
    ),
]
for label, item, qa, uat in examples:
    qs = ra.calculate_quality_score(item, qa, uat)
    print(f"{label:>14}  score={qs.score:>5.1f}  breakdown={qs.score_breakdown}")

# Floor at 0 — no negative scores leak.
qs = ra.calculate_quality_score(
    {"id": "X", "fix_subtask_count": 100, "qa_cycle_count": 100},
    [{"status": "changes_required"}] * 100,
    [{"overall_status": "changes_required"}],
)
assert qs.score == 0.0, "score floor at 0 violated"

## Priority queue (skill 13 — RISK-02)

Sort: score **descending**, ties broken by `updatedAt` **ascending** (oldest first). The tiebreaker matters when two clean tasks both score 100 — the one updated longer ago wins.


In [ ]:
linear_state = {
    "LIN-A": {
        "id": "LIN-A",
        "fix_subtask_count": 0,
        "qa_cycle_count": 0,
        "qa_history": [],
        "uat_results": [],
        "updatedAt": "2026-04-01T10:00:00Z",
    },
    "LIN-B": {
        "id": "LIN-B",
        "fix_subtask_count": 0,
        "qa_cycle_count": 0,
        "qa_history": [],
        "uat_results": [],
        "updatedAt": "2026-04-02T10:00:00Z",
    },
    "LIN-C": {
        "id": "LIN-C",
        "fix_subtask_count": 2,
        "qa_cycle_count": 1,
        "qa_history": [{"status": "changes_required"}],
        "uat_results": [],
        "updatedAt": "2026-03-15T10:00:00Z",
    },
}
q = ra.get_priority_queue(["LIN-A", "LIN-B", "LIN-C"], linear_state)
print("items =", q.items, "\nscores =", q.scores)
# A (100, older) before B (100, newer); C trails (lower score).
assert q.items[0] == "LIN-A"
assert q.items[1] == "LIN-B"
assert q.items[2] == "LIN-C"

## EPIC aggregation

Weighted average where weight = `max(1, qa_failures + fix_subtask_count)`. An EPIC of clean tasks averages 100 because each gets weight 1. An EPIC with one disastrous task pulls hard.


In [ ]:
scores = []
for spec in [
    {"id": "A", "fix_subtask_count": 0, "qa_cycle_count": 0},
    {"id": "B", "fix_subtask_count": 0, "qa_cycle_count": 0},
    {"id": "C", "fix_subtask_count": 10, "qa_cycle_count": 5},
]:
    scores.append(
        ra.calculate_quality_score(
            spec,
            [{"status": "changes_required"}] * spec["qa_cycle_count"],
            [],
        )
    )
epic = ra.calculate_epic_score(scores)
print("EPIC weighted score =", epic)
# Empty list returns 85.0 default — the deliberate 'unknown' value.
assert ra.calculate_epic_score([]) == 85.0

## Risk bands

>=75 = low, >=50 = medium, else high. Inspect the inflection points.


In [ ]:
for s in [100, 75.0, 74.999, 50.0, 49.999, 0]:
    print(f"{s:>8}  ->  {RiskAgent.risk_level(s)}")

## Global Orchestrator — ready-task filter (GORD-01/02)

Inputs: a list of plain dicts; the orchestrator keeps only `status='todo'` items whose `dependencies` are all in the `done` set. We exercise the public class method `_filter_ready_items` directly — it's stateless.


In [ ]:
from hsb.agents.global_orchestrator import GlobalOrchestrator

go = GlobalOrchestrator()
items = [
    {"id": "LIN-1", "status": "done", "dependencies": []},
    {"id": "LIN-2", "status": "todo", "dependencies": []},
    {"id": "LIN-3", "status": "todo", "dependencies": ["LIN-1"]},  # unblocked
    {"id": "LIN-4", "status": "todo", "dependencies": ["LIN-2"]},  # blocked
    {"id": "LIN-5", "status": "in_progress", "dependencies": []},  # not todo
]
ready = go._filter_ready_items(items)
ready_ids = sorted(t["id"] for t in ready)
print("ready =", ready_ids)
assert ready_ids == ["LIN-2", "LIN-3"]

## EPIC completion signal (GORD-04)


In [ ]:
# Empty children -> never ready.
assert go._check_epic_complete([{"id": "E", "type": "epic", "status": "todo"}]) is False
# All children done + qa_approved -> ready.
items = [
    {"id": "E", "type": "epic", "status": "in_progress"},
    {"id": "T1", "type": "task", "status": "done", "qa_status": "approved"},
    {"id": "T2", "type": "task", "status": "done", "qa_status": "not_required"},
]
assert go._check_epic_complete(items) is True
# One child still in progress -> not ready.
items[1]["status"] = "in_progress"
assert go._check_epic_complete(items) is False
print("GORD-04: epic readiness signal behaves correctly across cases")

## G10 — UAT pre-persist validation

Before any Linear write, the orchestrator runs `_uat_passes_g10`:
- **B1 coverage**: scenarios cover every AC (`AC-1..AC-N`).
- **B3 banned tokens**: scope-creep words like `refactor`, `code quality`, `naming`, `style`, `linter`, etc. fail the predicate.


In [ ]:
ac = ["User can log in", "Errors show inline"]
ok = UATResult(
    user_story_id="US-1",
    overall_status="approved",
    uat_cycle=1,
    scenarios=[
        UATScenario(
            criterion_id="AC-1",
            criterion_text=ac[0],
            status="pass",
            evidence="Logged in with valid creds and saw dashboard",
        ),
        UATScenario(
            criterion_id="AC-2",
            criterion_text=ac[1],
            status="pass",
            evidence="Submitted invalid form, error rendered inline",
        ),
    ],
)
assert _uat_passes_g10(ok, ac) is True

# Coverage gap (only AC-1)
gap = ok.model_copy(update={"scenarios": ok.scenarios[:1]})
assert _uat_passes_g10(gap, ac) is False

# Banned token in finding
creep = ok.model_copy(
    update={
        "scenarios": [
            ok.scenarios[0],
            UATScenario(
                criterion_id="AC-2",
                criterion_text=ac[1],
                status="fail",
                evidence="Form looked off",
                finding="Should refactor naming convention here",
            ),
        ]
    }
)
assert _uat_passes_g10(creep, ac) is False
print("G10: coverage gap and scope-creep both rejected")

## Free-form play

Build a synthetic Linear snapshot, run the queue, see what the orchestrator would dispatch.


In [ ]:
snapshot = {
    "LIN-100": {
        "id": "LIN-100",
        "status": "todo",
        "dependencies": [],
        "fix_subtask_count": 0,
        "qa_cycle_count": 0,
        "qa_history": [],
        "uat_results": [],
        "updatedAt": "2026-05-01T00:00:00Z",
    },
    "LIN-101": {
        "id": "LIN-101",
        "status": "todo",
        "dependencies": [],
        "fix_subtask_count": 1,
        "qa_cycle_count": 2,
        "qa_history": [{"status": "changes_required"}] * 2,
        "uat_results": [],
        "updatedAt": "2026-04-01T00:00:00Z",
    },
}
ready = go._filter_ready_items(list(snapshot.values()))
q = ra.get_priority_queue([t["id"] for t in ready], snapshot)
print("order:", q.items, "\nscores:", q.scores)